<a href="https://colab.research.google.com/github/chifa2003/BIGDATA/blob/main/TP01BigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ------------------------------------------------------------------------------
# 1. Wikipedia configuration   #تحديد روابط Wikipedia
# ------------------------------------------------------------------------------

BASE_URL = "https://en.wikipedia.org"   #نستعمله لبناء روابط المقالات.
API_URL = "https://en.wikipedia.org/w/api.php"  #هذا هو Wikipedia API.


#هذا يخلي الطلب يشبه طلب صادر من متصفح حقيقي مثل Chrome
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "Chrome/154.0.0.0 Safari/537.36"
    )
}

 #تحديد 1000 صف
TARGET_ROWS = 1000


#إنشاء أماكن لتخزين البيانات
scraped_data = []    #هذه قائمة راح نخزن فيها البيانات المستخرجة
seen = set()  #هذه نستعملها باش نمنع التكرار
pages_checked = 0   #هذا عداد لعدد الصفحات التي فحصناها

print("=" * 70)
print("TP01 - PYTHON WEB SCRAPING USING WIKIPEDIA")
print("=" * 70)

# ------------------------------------------------------------------------------
# 2. Question 1: Fetch a Wikipedia page
# ------------------------------------------------------------------------------

test_url = "https://en.wikipedia.org/wiki/Artificial_intelligence"      #تجربة تحميل صفحة Wikipedia

response = requests.get(      #هنا requests يروح للموقع ويحمل الصفحة.
    test_url,
    headers=headers,
    timeout=20
)

print("\n[Step 1] Fetching a Wikipedia page")
print("Status Code:", response.status_code)

# Display HTML source code     #عرض جزء من HTML
print("\n--- HTML SOURCE CODE PREVIEW ---")
print(response.text[:1000])     #يعني أول 1000 حرف
print("--------------------------------")

# ------------------------------------------------------------------------------
# 3. Question 2: Parse HTML using BeautifulSoup
# ------------------------------------------------------------------------------

#عني BeautifulSoup قامت بتحليل HTML وأصبح بإمكاننا البحث داخله.
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print("\n[Step 2] BeautifulSoup initialized successfully")

# Extract page title    #استخراج عنوان الصفحة
title = soup.find("h1")

if title:
    print("Page title:", title.get_text(" ", strip=True))  #نستخرج النص الموجود داخل العنصر.

# Find the main content area
main_content = soup.select_one("#mw-content-text")  #ابحث داخل HTML عن العنصر الذي عنده ID اسمه mw-content-text.

if main_content:
    test_paragraphs = main_content.find_all("p")  #أعطيني جميع عناصر <p> الموجودة داخل محتوى المقال.
else:
    test_paragraphs = []

print(
    "Paragraphs found on test page:",
    len(test_paragraphs)
)

# Display first few paragraphs
for paragraph in test_paragraphs[:3]:   #مرّ على أول 3 فقرات فقط

    text = paragraph.get_text(   #نستخرج النص فقط، بدون HTML.
        " ",
        strip=True
    )

    if text:
        print("\n", text[:300])

# ------------------------------------------------------------------------------
# 4. Collect 1000 real rows from Wikipedia
# ------------------------------------------------------------------------------

print("\n[Step 3] Starting web scraping...")
print("Target:", TARGET_ROWS, "real rows")
print("-" * 70)

while len(scraped_data) < TARGET_ROWS:    #طالما عدد الصفوف أقل من 1000، استمر في العمل.

    # --------------------------------------------------------------------------
    # Get random Wikipedia article pages
    # --------------------------------------------------------------------------
#يعني البرنامج يقول لـWikipedia:
#أعطيني 10 مقالات عشوائية.

    params = {
        "action": "query",
        "format": "json",
        "list": "random",
        "rnnamespace": 0,
        "rnlimit": 10
    }

    try:
#لطلب قائمة صفحات عشوائية من Wikipedia API. استعمال requests
        api_response = requests.get(
            API_URL,
            params=params,
            headers=headers,
            timeout=20
        )

        if api_response.status_code != 200:
            print(
                "API error:",
                api_response.status_code
            )
            continue

        random_pages = (
            api_response
            .json()["query"]["random"]
        )

    except Exception as e:

        print("API request error:", e)
        continue

    # --------------------------------------------------------------------------
    # Visit each Wikipedia page   #المرور على كل صفحة
    # --------------------------------------------------------------------------

    for page in random_pages:

        if len(scraped_data) >= TARGET_ROWS:
            break

        page_title = page["title"]   #نأخذ اسم الصفحة

 #بناء رابط الصفحة
        page_url = (
            BASE_URL
            + "/wiki/"
            + page_title.replace(" ", "_")
        )

        try:

            # Fetch page    #  البرنامج يدخل إلى صفحة Wikipedia ويحصل على HTML.
            page_response = requests.get(
                page_url,
                headers=headers,
                timeout=20
            )

            pages_checked += 1   #يزيد عداد الصفحات.

            if page_response.status_code != 200:
                continue

            # ------------------------------------------------------------------
            # Parse page using BeautifulSoup
            # ------------------------------------------------------------------

            page_soup = BeautifulSoup(   #نأخذ HTML ونحلله باستخدام BeautifulSoup
                page_response.text,
                "html.parser"
            )

            # Find main content
            main_content = page_soup.select_one(   #نبحث عن المحتوى الرئيسي.
                "#mw-content-text"
            )

            if not main_content:
                continue

            # Find paragraphs
            page_paragraphs = main_content.find_all("p")  #نأخذ جميع الفقرات.

            # ------------------------------------------------------------------
            # Extract paragraphs
            # ------------------------------------------------------------------

            for paragraph in page_paragraphs:

                text = paragraph.get_text(
                    " ",
                    strip=True
                )

                # Ignore empty or very short text    #إذا كانت الفقرة قصيرة جدًا، تجاهلها.
                if len(text) < 50:
                    continue

#هل هذه الصفحة وهذه الفقرة سبق أن خزناهما؟
                # Create unique key
                unique_key = (
                    page_title,
                    text
                )

                # Avoid duplicate data
                if unique_key in seen:
                    continue   #اذا نعم يتجاوزها

                seen.add(unique_key)  #اذا لا يضيفها للبيانات

                # Save extracted data   #تخزين البيانات

                scraped_data.append({   #هنا كل فقرة تصبح row واحد.
                    "Page_Title": page_title,
                    "Text": text,
                    "URL": page_url
                })

                if len(scraped_data) >= TARGET_ROWS:
                    break

            # Display progress
            if pages_checked % 10 == 0:

                print(
                    f"Pages checked: {pages_checked} | "
                    f"Rows collected: {len(scraped_data)}"
                )

            # Small delay
            time.sleep(0.2)

        except Exception as e:

            pages_checked += 1

            print(
                f"Error processing {page_title}: {e}"
            )

# ------------------------------------------------------------------------------
# 5. Create DataFrame
# ------------------------------------------------------------------------------

df = pd.DataFrame(    #تحويل البيانات إلى DataFrame
    scraped_data
)

print("\n" + "=" * 70)
print("SCRAPING COMPLETED")
print("=" * 70)

print(
    "Pages checked:",
    pages_checked
)

print(
    "Rows collected:",
    len(df)
)

# ------------------------------------------------------------------------------
# 6. Display first 10 rows
# ------------------------------------------------------------------------------

print("\n--- FIRST 10 ROWS ---")    #عرض أول 10 صفوف

display(
    df.head(10)
)

# ------------------------------------------------------------------------------
# 7. Check missing values
# ------------------------------------------------------------------------------

print("\n--- MISSING VALUES ---")

print(
    df.isnull().sum()   #هذا يفحص إذا عندنا خلايا فارغة
)

# ------------------------------------------------------------------------------
# 8. Check duplicate rows
# ------------------------------------------------------------------------------

print("\n--- DUPLICATE ROWS ---")

print(
    "Duplicate rows:",   # فحص التكرار
    df.duplicated().sum()
)

# ------------------------------------------------------------------------------
# 9. Verify 1000-row requirement
# ------------------------------------------------------------------------------

print("\n--- REQUIREMENT CHECK ---")

if len(df) >= 1000:

    print(
        "SUCCESS: At least 1000 real rows collected."
    )

else:

    print(
        "WARNING: Less than 1000 rows collected."
    )

# ------------------------------------------------------------------------------
# 10. Save to CSV
# ------------------------------------------------------------------------------

csv_filename = "wikipedia_scraping_1000_rows.csv"

df.to_csv(
    csv_filename,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nCSV file saved:",
    csv_filename
)

# ------------------------------------------------------------------------------
# 11. Verify CSV
# ------------------------------------------------------------------------------

df_check = pd.read_csv(
    csv_filename,
    encoding="utf-8-sig"
)

print(
    "Rows in CSV:",
    len(df_check)
)

# ------------------------------------------------------------------------------
# 12. Download CSV in Google Colab
# ------------------------------------------------------------------------------

from google.colab import files

files.download(
    csv_filename
)

# ------------------------------------------------------------------------------
# 13. Final result
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print("Website: Wikipedia")
print("Pages checked:", pages_checked)
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("CSV file:", csv_filename)

if len(df) >= 1000:

    print(
        "✅ TP REQUIREMENT PASSED"
    )

else:

    print(
        "❌ TP REQUIREMENT NOT PASSED"
    )

TP01 - PYTHON WEB SCRAPING USING WIKIPEDIA

[Step 1] Fetching a Wikipedia page
Status Code: 200

--- HTML SOURCE CODE PREVIEW ---
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Artificial intelligence - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-head

,Page_Title,Text,URL
0,The Crystal Maze,The Crystal Maze is a British game show devise...,https://en.wikipedia.org/wiki/The_Crystal_Maze
1,The Crystal Maze,The Crystal Maze originally consisted of six s...,https://en.wikipedia.org/wiki/The_Crystal_Maze
2,The Crystal Maze,"A 10-episode American version of the show, fil...",https://en.wikipedia.org/wiki/The_Crystal_Maze
3,The Crystal Maze,"In March 2016, The Crystal Maze Live Experienc...",https://en.wikipedia.org/wiki/The_Crystal_Maze
4,The Crystal Maze,The Crystal Maze ' s origins can be traced to ...,https://en.wikipedia.org/wiki/The_Crystal_Maze
5,The Crystal Maze,In order to create a more successful format fo...,https://en.wikipedia.org/wiki/The_Crystal_Maze
6,The Crystal Maze,Each team that competes on The Crystal Maze un...,https://en.wikipedia.org/wiki/The_Crystal_Maze
7,The Crystal Maze,Each game presents the potential of being lock...,https://en.wikipedia.org/wiki/The_Crystal_Maze
8,The Crystal Maze,There are two ways a lock-in can occur in The ...,https://en.wikipedia.org/wiki/The_Crystal_Maze
9,The Crystal Maze,"Once the team arrives at the Crystal Dome, the...",https://en.wikipedia.org/wiki/The_Crystal_Maze



--- MISSING VALUES ---
Page_Title    0
Text          0
URL           0
dtype: int64

--- DUPLICATE ROWS ---
Duplicate rows: 0

--- REQUIREMENT CHECK ---
SUCCESS: At least 1000 real rows collected.

CSV file saved: wikipedia_scraping_1000_rows.csv
Rows in CSV: 1000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


FINAL RESULT
Website: Wikipedia
Pages checked: 131
Rows: 1000
Columns: ['Page_Title', 'Text', 'URL']
CSV file: wikipedia_scraping_1000_rows.csv
✅ TP REQUIREMENT PASSED
